In [2]:
import os
import numpy as np
import pandas as pd

In [3]:
data_dir = "./data/METR-LA/"

train_file = os.path.join(data_dir, "train.npz")
valid_file = os.path.join(data_dir, "val.npz")
test_file = os.path.join(data_dir, "test.npz")

In [4]:
train_data = np.load(train_file, allow_pickle=True)
train_data

NpzFile './data/METR-LA/train.npz' with keys: x, y, x_offsets, y_offsets

In [5]:
train_data['x'].shape, train_data['y'].shape

((23974, 12, 207, 2), (23974, 12, 207, 2))

In [9]:
train_data['x'][0, :, 0, 0]

array([64.375     , 62.66666667, 64.        ,  0.        ,  0.        ,
       57.33333333, 66.5       , 63.625     , 68.75      , 63.5       ,
       65.22222222, 62.25      ])

In [8]:
train_data['y'][0, :, 0, 0]

array([61.125     , 58.55555556, 63.625     , 66.77777778, 55.875     ,
       64.33333333, 63.88888889, 63.125     , 62.125     , 61.5       ,
       63.22222222, 65.        ])

In [6]:
%load_ext autoreload
%autoreload 2

In [7]:
import torch
import torch.nn as nn
import time
import pandas as pd 
import numpy as np
import util
from engine import trainer

In [8]:
# data_dir = "./data/temp_dhw/"
data_dir = "./data/METR-LA/"
batch_size = 16
shuffle = True
epochs = 20

aptonly = True
addaptadj = True
randomadj = True
adjinit = None

in_dim = 2
seq_length = 12 # 365
num_nodes = 207 #3
nhid = 32
dropout = 0.3
learning_rate = 0.001
weight_decay = 0.0001
gcn_bool = True

device = torch.device('cuda')
print_every = 2
save = './garage.metr'

In [9]:
dataloader = util.load_dataset(data_dir, batch_size, batch_size, batch_size)
scaler = dataloader['scaler']
supports = None


In [10]:
engine = trainer(scaler, in_dim, seq_length, num_nodes, nhid, dropout,
                         learning_rate, weight_decay, device, supports, gcn_bool, addaptadj,
                         adjinit)

In [11]:
for iter, (x, y) in enumerate(dataloader["train_loader"].get_iterator()):
    trainx = torch.Tensor(x).to(device)  # (batch_size, seq_length, num_nodes, in_dim)
    trainy = torch.Tensor(y).to(device)

    trainx = trainx.transpose(1, 3)  # (batch_size, in_dim, seq_length, num_nodes)
    # trainy = trainy.transpose(1, 3)
    # metrics = engine.train(trainx, trainy[:,0,:,:])
    print(f"trainx.shape: {trainx.shape}, trainy.shape: {trainy.shape}")

    engine.model.train()
    engine.optimizer.zero_grad()
    input = nn.functional.pad(trainx, (1, 0, 0, 0))
    print(f"padded input shape: {input.shape}")

    output = engine.model(input)
    print(f"output shape: {output.shape}")

    output = output.transpose(1, 3)
    print(f"output transposed shape: {output.shape}")
    # output = [batch_size,12,num_nodes,1]
    # real = torch.unsqueeze(real_val,dim=1)

    predict = engine.scaler.inverse_transform(output)
    print(f"predict shape: {predict.shape}")

    # loss = engine.loss(predict, real, 0.0)
    # loss.backward()
    # if engine.clip is not None:
    #     torch.nn.utils.clip_grad_norm_(engine.model.parameters(), engine.clip)
    # engine.optimizer.step()
    # mape = util.masked_mape(predict,real,0.0).item()
    # rmse = util.masked_rmse(predict,real,0.0).item()
    break

trainx.shape: torch.Size([16, 2, 207, 12]), trainy.shape: torch.Size([16, 12, 207, 2])
padded input shape: torch.Size([16, 2, 207, 13])
1 start_conv x shape: torch.Size([16, 32, 207, 13])
Block 0: filter shape: torch.Size([16, 32, 207, 12])
Block 0: filter after tanh shape: torch.Size([16, 32, 207, 12])
Block 0: gate shape: torch.Size([16, 32, 207, 12])
Block 0: gate after sigmoid shape: torch.Size([16, 32, 207, 12])
Block 0: x after filter*gate shape: torch.Size([16, 32, 207, 12])
Block 1: filter shape: torch.Size([16, 32, 207, 10])
Block 1: filter after tanh shape: torch.Size([16, 32, 207, 10])
Block 1: gate shape: torch.Size([16, 32, 207, 10])
Block 1: gate after sigmoid shape: torch.Size([16, 32, 207, 10])
Block 1: x after filter*gate shape: torch.Size([16, 32, 207, 10])
Block 2: filter shape: torch.Size([16, 32, 207, 9])
Block 2: filter after tanh shape: torch.Size([16, 32, 207, 9])
Block 2: gate shape: torch.Size([16, 32, 207, 9])
Block 2: gate after sigmoid shape: torch.Size([16

In [12]:
predict[0,:,0]

tensor([[54.5960, 53.7458, 55.9527, 55.9091, 53.9624, 55.4629, 54.8190, 55.1971,
         54.2148, 54.7362, 53.0143, 53.9911]], device='cuda:0',
       grad_fn=<SelectBackward0>)